In [ ]:
import plotly.graph_objects as go
import pandas as pd
df = pd.read_csv('results.csv')

In [ ]:
ranking_df = df.groupby('Model')['F1'].mean().reset_index()
ranking_df = ranking_df.sort_values(by='F1', ascending=False).reset_index(drop=True)

# 3. กำหนดชุดสีตามที่คุณระบุ (เรียงตามอันดับโมเดล)
_MODEL_COLORS = [
    "#2563eb", # blue
    "#dc2626", # red
    "#16a34a", # green
    "#d97706", # amber
    "#7c3aed", # violet
    "#0891b2", # cyan
    "#db2777", # pink
    "#65a30d", # lime
]

# แมตช์สีกับโมเดล (ใช้แค่เท่าที่มีจำนวนโมเดล)
model_colors = _MODEL_COLORS[:len(ranking_df)]

# พลิกข้อมูลเพื่อพล็อต (Plotly จะวาดจากล่างขึ้นบนในแนวนอน)
df_plot = ranking_df.iloc[::-1].reset_index(drop=True)
plot_colors = model_colors[::-1]

# 4. สร้างกราฟ
fig = go.Figure(go.Bar(
    x=df_plot['F1'],
    y=df_plot['Model'],
    orientation='h',
    marker=dict(
        color=plot_colors,
        opacity=0.85, # ปรับความอ่อนเพื่อให้ดูนุ่มนวลขึ้น
        line=dict(color='white', width=1)
    ),
    text=df_plot['F1'].apply(lambda x: f'<b>{x:.2f}</b>'),
    textposition='inside',
    insidetextanchor='end',
    textfont=dict(size=14, color='white'),
    width=0.7 # ความหนาของแท่ง
))

# 5. ปรับแต่ง Layout ให้คล้ายธีมในรูปภาพ
fig.update_layout(
    title=dict(
        text="<b>Model Performance Ranking</b><br><span style='font-size:14px; color:#6b7280;'>Mean F1-score comparison across all models</span>",
        font=dict(size=22, color='#111827'),
        x=0.03,
        y=0.92
    ),
    xaxis=dict(
        title="Mean F1-score (%)",
        showgrid=True,
        gridcolor='#f3f4f6', # เส้น Grid จางๆ
        range=[0, 105],
        dtick=10,
        zeroline=False
    ),
    yaxis=dict(
        title="",
        showgrid=False,
        tickfont=dict(size=14, color='#374151')
    ),
    template="plotly_white",
    margin=dict(l=160, r=40, t=100, b=60),
    height=550,
    plot_bgcolor='white',
    paper_bgcolor='white',
    bargap=0.2
)

# แสดงผล
fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go
import random

# --- [FIX: Repeated Division & Out of Bounds Bug] ---
# ใช้ copy เพื่อป้องกันไม่ให้รันซ้ำแล้วค่าโดนหารไปเรื่อยๆ
plot_df = df.copy()

# ตรวจสอบและหาร 100 เฉพาะเมื่อค่าเป็นหลักหน่วย/สิบ (ป้องกันการรันซ้ำใน Cell เดียวกัน)
if plot_df['Precision'].max() > 1.05:
    plot_df['Precision'] = plot_df['Precision'] / 100
if plot_df['Recall'].max() > 1.05:
    plot_df['Recall'] = plot_df['Recall'] / 100

# 1. กำหนดค่าพื้นฐาน
_MODEL_COLORS = ["#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2", "#db2777", "#65a30d"]
_TH_SYMBOLS = {
    "P99 Static": "circle",
    "Sliding Mu+αStd": "diamond",
    "Adaptive-z": "square",
    "Entropy-lock": "x"
}
_TH_SHORT = {
    "P99 Static": "P99",
    "Sliding Mu+αStd": "Slid",
    "Adaptive-z": "Adpt",
    "Entropy-lock": "Entr"
}

fig = go.Figure()

# 2. F1-Isolines (เส้นโค้งพื้นหลัง)
x_iso = np.linspace(0.01, 1, 100)
for f1 in [0.2, 0.4, 0.6, 0.8]:
    # Precision = (f1 * Recall) / (2 * Recall - f1)
    y_iso = (f1 * x_iso) / (2 * x_iso - f1)
    mask = (y_iso >= 0) & (y_iso <= 1)
    
    rx = x_iso[mask]
    py = y_iso[mask]
    
    if len(rx) > 0:
        fig.add_trace(go.Scatter(
            x=rx, y=py,
            mode="lines",
            line=dict(color="#cbd5e1", width=1, dash="dot"),
            showlegend=False, hoverinfo="skip"
        ))
        
        # [FIX: Dynamic Indexing] เลือกจุดกึ่งกลางของเส้นเพื่อแปะป้าย F1 ป้องกัน IndexError
        label_idx = len(rx) // 2 
        fig.add_annotation(
            x=rx[label_idx], y=py[label_idx],
            text=f"F1={f1}", showarrow=False,
            font=dict(size=9, color="#94a3b8"),
            bgcolor="white", opacity=0.8
        )

# 3. Best Zone Shading & Annotation
fig.add_shape(
    type="rect", x0=0.75, y0=0.75, x1=1.0, y1=1.0,
    fillcolor="rgba(34,197,94,0.06)",
    line=dict(color="rgba(34,197,94,0.25)", width=1, dash="dot"),
    layer="below"
)
fig.add_annotation(
    x=0.875, y=0.88,
    text="✦ Best Zone",
    showarrow=False,
    font=dict(size=10, color="rgba(22,163,74,0.8)", weight="bold"),
    xanchor="center",
    opacity=0.6
)

# 4. วาดจุด (ไม่มี Text ในกราฟตามที่ต้องการ)
models = plot_df['Model'].unique()
for i, model_name in enumerate(models):
    model_data = plot_df[plot_df['Model'] == model_name]
    color = _MODEL_COLORS[i % len(_MODEL_COLORS)]
    
    # วนลูปตาม Threshold เพื่อให้สัญลักษณ์ตรงตามที่กำหนด
    for th_key in _TH_SYMBOLS.keys():
        row = model_data[model_data['Threshold'] == th_key]
        if row.empty: continue
        
        row = row.iloc[0]
        # Jitter เล็กน้อยเพื่อให้เห็นขอบขาวเวลาทับกันเป๊ะๆ
        display_x = row['Recall'] + random.uniform(-0.002, 0.002)
        display_y = row['Precision'] + random.uniform(-0.002, 0.002)
        
        fig.add_trace(go.Scatter(
            x=[display_x],
            y=[display_y],
            mode="markers", # เอา +text ออกแล้ว
            name=f"{model_name} · {_TH_SHORT.get(th_key)}",
            legendgroup=model_name,
            legendgrouptitle=dict(text=model_name) if th_key == "P99 Static" else None,
            marker=dict(
                size=14, 
                color=color, 
                symbol=_TH_SYMBOLS.get(th_key),
                opacity=0.9,
                line=dict(width=1.5, color="white")
            ),
            hovertemplate=(
                f"<b>{model_name}</b><br>"
                f"TH: {th_key}<br>"
                f"Recall: %{{x:.3f}}<br>"
                f"Precision: %{{y:.3f}}<extra></extra>"
            )
        ))

# 5. Layout Setup
fig.update_layout(
    height=550,
    template="plotly_white",
    title=dict(
        text="<b>PR Plot — Precision × Recall</b>",
        font=dict(size=16, color="#0f172a"),
        x=0.02
    ),
    xaxis=dict(
        title="Recall", range=[-0.02, 1.05],
        gridcolor="#f1f5f9", tickformat=".0%", zeroline=False
    ),
    yaxis=dict(
        title="Precision", range=[-0.02, 1.05],
        gridcolor="#f1f5f9", tickformat=".0%", zeroline=False
    ),
    legend=dict(
        tracegroupgap=8, 
        itemsizing='constant',
        font=dict(size=10),
        bordercolor="#e2e8f0", borderwidth=1,
        x=1.02, y=1
    ),
    margin=dict(l=60, r=200, t=80, b=80)
)

# สัญลักษณ์ Legend อ้างอิงด้านล่าง
symbol_hint = " &nbsp;·&nbsp; ".join([f"<b>{v.upper()}</b> = {k}" for k, v in _TH_SYMBOLS.items()])
fig.add_annotation(
    text=symbol_hint, xref="paper", yref="paper",
    x=0, y=-0.15, showarrow=False,
    font=dict(size=10, color="#64748b"), xanchor="left"
)

fig.show()

In [ ]:
import pandas as pd
from dash import Dash, dash_table, html, dcc, Input, Output, State
from dash.dash_table.Format import Format, Scheme
import io

df['Detection'] = df['Seg_Detected'].astype(str) + " / " + df['Seg_Total'].astype(str)
percent_cols = ['Precision', 'Recall', 'F1', 'Accuracy'] + [c for c in df.columns if 'F1_NOV' in c]

# 2. ฟังก์ชัน In-cell Bar (สีเขียวอ่อน)
def data_bars(df, column):
    styles = []
    for i in range(1, 21):
        bound = i * 5
        styles.append({
            'if': {'filter_query': f'{{{column}}} >= {bound-5}', 'column_id': column},
            'background': f'linear-gradient(90deg, #dcfce7 0%, #dcfce7 {bound}%, white {bound}%, white 100%)',
            'paddingBottom': 8, 'paddingTop': 8
        })
    return styles

_MODEL_COLORS = ["#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2", "#db2777", "#65a30d"]
model_color_map = {m: _MODEL_COLORS[i % len(_MODEL_COLORS)] for i, m in enumerate(df['Model'].unique())}

# 3. เริ่ม App (เพิ่ม html2canvas สำหรับโหลด PNG)
app = Dash(__name__, external_scripts=["https://cdnjs.cloudflare.com/ajax/libs/html2canvas/1.4.1/html2canvas.min.js"])

app.layout = html.Div([
    # Header & Buttons
    html.Div([
        html.Div([
            html.H2("Model Performance Matrix", style={'fontFamily': 'sans-serif', 'margin': '0', 'fontSize': '32px'}),
            html.P("In-cell visualization with full metric exports", style={'color': '#64748b', 'fontFamily': 'sans-serif'})
        ], style={'flex': '1'}),
        
        html.Div([
            html.Button("🖼️ Download PNG", id="btn-download-png", 
                        style={'backgroundColor': '#16a34a', 'color': 'white', 'border': 'none', 
                               'padding': '12px 24px', 'borderRadius': '6px', 'cursor': 'pointer',
                               'fontWeight': 'bold', 'marginRight': '10px'}),
            html.Button("📥 Download CSV", id="btn-download-csv", 
                        style={'backgroundColor': '#2563eb', 'color': 'white', 'border': 'none', 
                               'padding': '12px 24px', 'borderRadius': '6px', 'cursor': 'pointer',
                               'fontWeight': 'bold'}),
            dcc.Download(id="download-csv"),
            html.Div(id="png-status") # สำหรับสถานะการโหลดรูป
        ], style={'display': 'flex'})
    ], style={'padding': '30px', 'borderBottom': '1px solid #e2e8f0', 'backgroundColor': 'white'}),

    # ตาราง (ใส่ id 'table-wrapper' เพื่อใช้ถ่ายรูป)
    html.Div([
        dash_table.DataTable(
            id='main-table',
            data=df.to_dict('records'),
            columns=[{
                "name": i.replace('_', ' ').upper(), "id": i, "type": "numeric" if i in percent_cols else "text",
                "format": Format(precision=1, scheme=Scheme.fixed).symbol_suffix(' %') if i in percent_cols else None
            } for i in ['Model', 'Threshold', 'Detection'] + [c for c in df.columns if c not in ['Model', 'Threshold', 'Detection', 'Seg_Detected', 'Seg_Total']]],
            sort_action="native",
            fixed_columns={'headers': True, 'data': 2},
            style_table={'minWidth': '100%'},
            style_cell={
                'textAlign': 'left', 'fontFamily': 'sans-serif', 'padding': '18px 20px', 
                'fontSize': '16px', # [ขยายฟอนต์ใหญ่มาก]
                'borderBottom': '1px solid #f1f5f9'
            },
            style_header={
                'backgroundColor': '#f8fafc', 'fontWeight': 'bold', 'color': '#475569', 
                'fontSize': '14px', 'borderBottom': '2px solid #e2e8f0'
            },
            style_data_conditional=[
                *[{
                    'if': {'filter_query': f'{{Model}} = "{m}"', 'column_id': 'Model'},
                    'color': color, 'fontWeight': 'bold', 'borderLeft': f'8px solid {color}'
                } for m, color in model_color_map.items()],
                *[style for col in percent_cols for style in data_bars(df, col)],
                {'if': {'column_id': percent_cols}, 'color': '#065f46', 'fontWeight': 'bold'}
            ],
        )
    ], id='table-container', style={'padding': '20px', 'backgroundColor': 'white'})
])

# 4. [CALLBACKS]
# ปุ่มโหลด CSV
@app.callback(Output("download-csv", "data"), Input("btn-download-csv", "n_clicks"), prevent_initial_call=True)
def download_csv(n_clicks):
    return dcc.send_data_frame(df.to_csv, "performance_results.csv", index=False)

# ปุ่มโหลด PNG (JavaScript ทำงานฝั่ง Browser)
app.clientside_callback(
    """
    function(n_clicks) {
        if(n_clicks > 0) {
            const table = document.getElementById('table-container');
            html2canvas(table, {backgroundColor: "#ffffff", scale: 2}).then(canvas => {
                const link = document.createElement('a');
                link.download = 'model_performance_summary.png';
                link.href = canvas.toDataURL("image/png");
                link.click();
            });
        }
        return "";
    }
    """,
    Output("png-status", "children"),
    Input("btn-download-png", "n_clicks"),
)

if __name__ == '__main__':
    app.run(debug=True)